# AutoML Tabular Baseline (Question A)

Tests whether a simple, non-graph tabular model can match the GNNs'
predictive performance on the same balanced case subset -- if it can, the
GNNs' ROC-AUC ceiling isn't coming from relational/graph structure; if it
clearly can't, the graph structure is earning its complexity. One row per
Case; one-to-many child tables (Task, CaseHistory) are collapsed into
fixed summary columns via group-by aggregation, since a tabular model
needs a fixed-width row per case.

## Setup

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import re
import torch
import pandas as pd
from pyspark.sql import functions as F

volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
AUTOML_DIR = f"{volume}automl_baseline/"
os.makedirs(AUTOML_DIR, exist_ok=True)

# base and sulo splits are identical (same TrainingData split, random_state=42) --
# using base's since it's smaller to load.
SPLIT_CACHE_PATH = f"{volume}transe/base_split.pt"

TOP_N_TASK_TYPES = 10
TOP_N_CHANGED_FIELDS = 20

def report_category_coverage(df, category_col, top_n, label):
    """Prints distinct-value count and top-N coverage % before bucketing into
    top-N + 'Other', so it's clear whether top_n is reasonable. Returns the top-N list."""
    counts = (df.filter(F.col(category_col).isNotNull())
        .groupBy(category_col).count().orderBy(F.desc('count')))
    total_distinct = counts.count()
    total_rows = counts.agg(F.sum('count').alias('n')).first()['n']
    top_rows = counts.limit(top_n).collect()
    top_list = [r[category_col] for r in top_rows]
    covered = sum(r['count'] for r in top_rows)
    coverage_pct = 100.0 * covered / total_rows if total_rows else 0.0
    print(f"[{label}] {total_distinct} distinct values, {total_rows:,} total rows")
    print(f"[{label}] Top {top_n} covers {covered:,} rows ({coverage_pct:.1f}%) -- "
          f"'Other' absorbs the remaining {total_rows - covered:,} ({100 - coverage_pct:.1f}%)")
    for r in top_rows:
        print(f"    {r[category_col]}: {r['count']:,} ({100.0 * r['count'] / total_rows:.1f}%)")
    return top_list


def sanitize_and_dedupe(values):
    """Maps each raw value to a Delta-safe column-name suffix (Delta rejects
    space/,/;/{}/()/newline/tab/= in column names), disambiguating if two
    different values sanitize to the same string."""
    seen = {}
    result = {}
    for v in values:
        base = re.sub(r'[^0-9a-zA-Z_]+', '_', str(v)).strip('_') or 'blank'
        name = base
        if name in seen:
            seen[name] += 1
            name = f"{base}_{seen[name]}"
        else:
            seen[name] = 0
        result[v] = name
    return result


## Balanced case set + case-level base columns + target label

Starts from `ds_ccc.kg_sulo_balanced_case_ids` (same table graph_sage.ipynb/RGCN.ipynb
use, not Graph_population.ipynb's broader unbalanced table) so the row set
matches the GNN experiments exactly. Raw case description/solution text is
not pulled here -- the precomputed BERT embeddings are used instead, further down.

In [ ]:
spark.sql("""
    SELECT case_id FROM ds_ccc.kg_sulo_balanced_case_ids
""").createOrReplaceTempView("sampled_case_ids")

case_base = spark.sql("""
    SELECT
        c.case_id, c.person_account_id, c.business_account_id, c.case_vehicle_structure_id,
        c.case_closure_type, c.case_type, c.case_service_type, c.support_level,
        c.customer_country, c.market_country, c.case_reason, c.case_priority,
        c.case_handling_time_sec, c.case_conversation_amt, c.case_session_amt, c.is_closed_flg,
        DATEDIFF(c.case_closed_date, c.case_created_date) AS case_duration_days
    FROM sampled_case_ids sc
    JOIN stg_ccc.w_case c ON sc.case_id = c.case_id
""")

# Target label -- identical construction (questionid filter, numbervalue >= 3
# threshold) to Graph_population.ipynb's own survey join, just scoped to the
# balanced case set instead of the original unbalanced sample.
survey_label_raw = spark.sql("""
    SELECT DISTINCT
        c.case_id,
        CASE WHEN CAST(sqr.numbervalue AS FLOAT) >= 3 THEN 1 ELSE 0 END AS satisfied
    FROM sampled_case_ids c
    JOIN stg_ccc.sf_survey_subject ss ON c.case_id = ss.subjectid
    JOIN stg_ccc.sf_survey_invitation si ON ss.parentid = si.id
    JOIN stg_ccc.sf_survey_question_response sqr ON si.id = sqr.invitationid
    WHERE sqr.questionid = '0Ku7S000000RnrrSAC'
    AND sqr.numbervalue IS NOT NULL
""")

# A small number of cases have more than one survey response to this
# question that disagree on the >=3 threshold (confirmed via a real
# duplicate-row investigation: 5 cases produced two rows each here, with
# every column identical except 'satisfied' itself). SELECT DISTINCT alone
# can't collapse these -- (case_id, satisfied) is a genuinely different
# pair for each conflicting response. No response timestamp was pulled
# here to defensibly pick "the real" answer, so these are dropped as
# ambiguous ground truth rather than guessed at.
_label_counts = survey_label_raw.groupBy('case_id').count()
_ambiguous_cases = _label_counts.filter(F.col('count') > 1).select('case_id')
_ambiguous_n = _ambiguous_cases.count()
if _ambiguous_n:
    print(f"Dropping {_ambiguous_n} case(s) with conflicting survey responses (ambiguous ground truth).")
survey_label = survey_label_raw.join(_ambiguous_cases, on='case_id', how='left_anti')

case_base = case_base.join(survey_label, on='case_id', how='inner')
print(f"case_base rows: {case_base.count():,} (expected ~53,569, minus any dropped ambiguous cases)")

## Account (two to-one joins, not an aggregation problem)

In [ ]:
account_cols = spark.sql("""
    SELECT id, salutation, type AS account_type, core_country__c AS account_country,
           core_preferredlanguage__c AS account_language
    FROM stg_ccc.sf_account
    WHERE isdeleted = 'N'
""")

person_account = (account_cols
    .select(*[F.col(c).alias(f"person_account_{c}") if c != 'id' else F.col(c).alias('person_account_id')
               for c in account_cols.columns]))
business_account = (account_cols
    .select(*[F.col(c).alias(f"business_account_{c}") if c != 'id' else F.col(c).alias('business_account_id')
               for c in account_cols.columns]))

case_base = (case_base
    .join(person_account, on='person_account_id', how='left')
    .join(business_account, on='business_account_id', how='left'))

## Vehicle (to-one join)

In [ ]:
vehicle_cols = spark.sql("""
    SELECT vehicle_structure_id, vehicle_structure_name, brand_name, class_name,
           serie_name, product_group_name
    FROM stg_ccc.w_vehicle_structure
""")

case_base = (case_base
    .withColumnRenamed('case_vehicle_structure_id', 'vehicle_structure_id')
    .join(vehicle_cols, on='vehicle_structure_id', how='left'))

## Task aggregation (1-to-many -> fixed columns)

`num_tasks`, high-priority count, call-duration aggregates, and a top-N
`task_type` count breakdown (rest bucketed as "Other" to keep column count fixed).

In [ ]:
task_base = spark.sql("""
    SELECT t.whatid AS case_id, t.type AS task_type, t.ishighpriority, t.calldurationinseconds
    FROM westeurope_cac_dwh_prd.sf_ccc.task t
    JOIN sampled_case_ids c ON t.whatid = c.case_id
""")

top_task_types = report_category_coverage(task_base, 'task_type', TOP_N_TASK_TYPES, 'task_type')

# '__UNCATEGORIZED__' (not the human-readable 'Other') is used as the
# catch-all bucket label -- the real data can and does contain a genuine
# task_type value literally called "Other", and if that value also makes
# top_task_types, appending a second, colliding 'Other' to the pivot's
# value list produces two identically-named output columns
# (Spark's [AMBIGUOUS_REFERENCE] error, reproduced against this exact data).
_TASK_OTHER_BUCKET = '__UNCATEGORIZED__'
task_pivot_values = top_task_types + [_TASK_OTHER_BUCKET]

task_base = task_base.withColumn(
    'task_type_bucket',
    F.when(F.col('task_type').isin(top_task_types), F.col('task_type')).otherwise(F.lit(_TASK_OTHER_BUCKET)),
)

task_type_counts = (task_base.groupBy('case_id')
    .pivot('task_type_bucket', task_pivot_values)
    .count())
_task_type_col_map = sanitize_and_dedupe(top_task_types)
for c in task_pivot_values:
    new_name = 'num_tasks_type_uncategorized' if c == _TASK_OTHER_BUCKET else f"num_tasks_type_{_task_type_col_map[c]}"
    task_type_counts = task_type_counts.withColumnRenamed(c, new_name)

task_numeric = task_base.groupBy('case_id').agg(
    F.count('*').alias('num_tasks'),
    F.sum(F.col('ishighpriority').cast('int')).alias('num_high_priority_tasks'),
    F.avg('calldurationinseconds').alias('mean_task_call_duration_sec'),
    F.max('calldurationinseconds').alias('max_task_call_duration_sec'),
    F.sum('calldurationinseconds').alias('sum_task_call_duration_sec'),
)

task_features = task_numeric.join(task_type_counts, on='case_id', how='left')
case_base = case_base.join(task_features, on='case_id', how='left')

## CaseHistory aggregation (1-to-many changelog -> fixed columns)

CaseHistory is a field-change audit log (one row per edit), not a set of
similar entities like Task. Same fixed-column technique: total count,
distinct-fields-touched count, a top-N `changed_field` breakdown, and
modification time span.

In [ ]:
history_base = spark.sql("""
    SELECT ch.caseid AS case_id, ch.field AS changed_field, ch.createddate AS event_date
    FROM westeurope_cac_dwh_prd.sf_ccc.casehistory ch
    JOIN sampled_case_ids c ON ch.caseid = c.case_id
""")

top_fields = report_category_coverage(history_base, 'changed_field', TOP_N_CHANGED_FIELDS, 'changed_field')

# Same collision risk/fix as the Task cell above -- changed_field could
# equally contain a real value literally called "Other".
_HISTORY_OTHER_BUCKET = '__UNCATEGORIZED__'
history_pivot_values = top_fields + [_HISTORY_OTHER_BUCKET]

history_base = history_base.withColumn(
    'field_bucket',
    F.when(F.col('changed_field').isin(top_fields), F.col('changed_field')).otherwise(F.lit(_HISTORY_OTHER_BUCKET)),
)

history_field_counts = (history_base.groupBy('case_id')
    .pivot('field_bucket', history_pivot_values)
    .count())
_history_field_col_map = sanitize_and_dedupe(top_fields)
for c in history_pivot_values:
    new_name = 'num_history_field_uncategorized' if c == _HISTORY_OTHER_BUCKET else f"num_history_field_{_history_field_col_map[c]}"
    history_field_counts = history_field_counts.withColumnRenamed(c, new_name)

history_numeric = history_base.groupBy('case_id').agg(
    F.count('*').alias('num_modifications'),
    F.countDistinct('changed_field').alias('num_distinct_fields_changed'),
    F.datediff(F.max('event_date'), F.min('event_date')).alias('modification_span_days'),
)

history_features = history_numeric.join(history_field_counts, on='case_id', how='left')
case_base = case_base.join(history_features, on='case_id', how='left')

## CaseDescription / CaseSolutionFinalStatement text embeddings

`ds_ccc.sulokg_embeddings_dv` columns confirmed: `case_id`,
`description_vector`, `solution_vector`.

In [ ]:
embedding_table = spark.sql("""
    SELECT e.*
    FROM sampled_case_ids c
    JOIN ds_ccc.sulokg_embeddings_dv e ON c.case_id = e.case_id
""")

In [ ]:
# Confirmed against ds_ccc.sulokg_embeddings_dv's actual schema
# (case_id, description_vector, solution_vector).
CASE_DESC_EMB_COL = 'description_vector'
CASE_SOLN_EMB_COL = 'solution_vector'
EMB_DIM = 1024  # matches FEATURE_DIM used throughout the rest of the pipeline

# Kept as ONE array column each -- not exploded into 1024 scalar columns.
# Keeps the saved Delta table small and readable, matches how
# ds_ccc.sulokg_embeddings_dv itself stores these, and avoids Delta
# column-naming restrictions entirely for this data. A model still needs
# each dimension as its own feature somewhere -- that expansion now
# happens in pandas right before XGBoost fitting instead (see the
# training section), not in the stored table.
case_desc_emb = embedding_table.select(F.col('case_id'), F.col(CASE_DESC_EMB_COL).alias('case_desc_emb'))
case_soln_emb = embedding_table.select(F.col('case_id'), F.col(CASE_SOLN_EMB_COL).alias('case_soln_emb'))

case_base = (case_base
    .join(case_desc_emb, on='case_id', how='left')
    .join(case_soln_emb, on='case_id', how='left'))

## Attach the shared train/val/test split

Reuses the exact case-level split already cached by transE.ipynb's
`build_case_split`, not a fresh `train_test_split` call -- re-running the
split on a differently-ordered case list does not reproduce the same held-out set.

In [ ]:
def _entity_id(uri):
    """Trailing path segment of a raw N-Triples URI, e.g.
    '<http://.../Case/500KC...>' -> '500KC...' -- the native Salesforce
    Case ID (string), never a numeric surrogate despite the column name.
    Do NOT int() this."""
    s = uri.strip('<>').strip('"')
    return s.split('/')[-1] if '/' in s else s

split_data = torch.load(SPLIT_CACHE_PATH, weights_only=False)

_split_name_map = {'train': 'train', 'val': 'validate', 'test': 'test'}
split_rows = []
for split_name, (uris, labels) in split_data.items():
    for uri in uris:
        split_rows.append((_entity_id(uri), _split_name_map[split_name]))

split_pdf = pd.DataFrame(split_rows, columns=['case_id', 'split'])
split_sdf = spark.createDataFrame(split_pdf)
print(split_pdf['split'].value_counts())

full_df = case_base.join(split_sdf, on='case_id', how='inner')  
_dropped = case_base.count() - full_df.count()
if _dropped:
    print(f"WARNING: {_dropped} case(s) had no entry in the cached split and were dropped.")

# Cases with zero tasks/history get NULL from the left joins, not 0 -- fix up.
count_like_cols = [c for c in full_df.columns if c.startswith('num_') or c.startswith('sum_')]
full_df = full_df.fillna(0, subset=count_like_cols)

print(f"Final table: {full_df.count():,} rows, {len(full_df.columns)} columns")

## Save the assembled table

Persisted so this notebook can resume from here (or load the table
directly, skipping the feature-engineering pipeline above) without
re-running every join/aggregation.

In [ ]:
AUTOML_TABLE_NAME = "ds_ccc.kg_automl_baseline_table"

full_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(AUTOML_TABLE_NAME)
print(f"Saved: {AUTOML_TABLE_NAME} ({full_df.count():,} rows, {len(full_df.columns)} columns)")

# No reload into full_df needed here -- the next section loads
# AUTOML_TABLE_NAME directly via spark.table(...).toPandas(). This also
# means a later re-run needing only the saved table can skip straight to
# that section without running anything in this notebook above it.

## Train directly (bypassing `automl.classify`)

`automl.classify()` OOM'd on this table's ~3,000+ columns (its driver-side
profiling step, not the data's actual size). PCA-reducing the embeddings to
fit would confound the comparison, so this trains XGBoost directly instead
-- keeps every embedding dimension intact, and its native categorical
support (`enable_categorical=True`) handles the string columns without
manual encoding.

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, average_precision_score,
    accuracy_score, precision_score, recall_score,
)

full_pdf = spark.table(AUTOML_TABLE_NAME).toPandas()

ID_COLS = ["case_id", "person_account_id", "business_account_id", "vehicle_structure_id"]
TARGET_COL = "satisfied"
EMBEDDING_COLS = ["case_desc_emb", "case_soln_emb"]

def expand_embedding_column(df, col, prefix, dim=EMB_DIM):
    """Turns a pandas column of array-likes (or None, for any case with no
    matching row in ds_ccc.sulokg_embeddings_dv) into `dim` separate float
    columns -- XGBoost needs each embedding dimension as its own feature,
    this does that expansion here in memory instead of in the stored Delta
    table, which is kept narrow/readable with a single array column."""
    stacked = np.vstack(df[col].apply(
        lambda v: np.asarray(v, dtype='float32') if v is not None else np.full(dim, np.nan, dtype='float32')
    ))
    return pd.DataFrame(stacked, columns=[f"{prefix}_{i}" for i in range(dim)], index=df.index)

desc_emb_df = expand_embedding_column(full_pdf, 'case_desc_emb', 'case_desc_emb')
soln_emb_df = expand_embedding_column(full_pdf, 'case_soln_emb', 'case_soln_emb')
full_pdf = pd.concat([full_pdf.drop(columns=EMBEDDING_COLS), desc_emb_df, soln_emb_df], axis=1)

feature_cols = [c for c in full_pdf.columns if c not in ID_COLS + [TARGET_COL, "split"]]

# XGBoost's enable_categorical needs these as pandas 'category' dtype, not
# plain strings/objects.
categorical_cols = [c for c in feature_cols if full_pdf[c].dtype == object]
for c in categorical_cols:
    full_pdf[c] = full_pdf[c].astype('category')
print(f"{len(feature_cols)} feature columns ({len(categorical_cols)} categorical, "
      f"{len(feature_cols) - len(categorical_cols)} numeric)")

train_df = full_pdf[full_pdf['split'] == 'train']
val_df = full_pdf[full_pdf['split'] == 'validate']
test_df = full_pdf[full_pdf['split'] == 'test']

X_train, y_train = train_df[feature_cols], train_df[TARGET_COL]
X_val, y_val = val_df[feature_cols], val_df[TARGET_COL]
X_test, y_test = test_df[feature_cols], test_df[TARGET_COL]
print(f"train={len(X_train):,} val={len(X_val):,} test={len(X_test):,}")

### Split-integrity check

A case_id leaking across more than one split would alone inflate test
metrics regardless of any feature-level leakage -- checked first, before
looking at what the model actually learned from.

In [ ]:
_split_counts_per_case = full_pdf.groupby('case_id')['split'].nunique()
_contaminated = _split_counts_per_case[_split_counts_per_case > 1]
if len(_contaminated) > 0:
    print(f"LEAK: {len(_contaminated)} case_id(s) appear in more than one split!")
    print(_contaminated.head(20))
else:
    print(f"OK: every case_id appears in exactly one split ({len(_split_counts_per_case):,} unique cases).")

_dupe_rows = full_pdf['case_id'].duplicated().sum()
print(f"Duplicate case_id rows in full_pdf: {_dupe_rows} (expect 0)")

In [ ]:
import time

model = XGBClassifier(
    tree_method='hist',
    enable_categorical=True,
    eval_metric='auc',
    early_stopping_rounds=20,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
)

t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
runtime_sec = time.time() - t0
print(f"Trained {model.best_iteration + 1} rounds (early-stopped from n_estimators=500) in {runtime_sec:.1f}s")

### Feature importance

Task/CaseHistory aggregates don't filter out events that happened after the
survey was submitted -- a heavy skew toward these features here is worth
checking against event timestamps before trusting the result.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print("Top 20 features by importance:")
print(importances.head(20).to_string())

_history_prefixes = ('num_modifications', 'num_distinct_fields_changed', 'modification_span_days', 'num_history_field_')
_history_importance = importances[importances.index.str.startswith(_history_prefixes)].sum()
print(f"\nTotal importance from CaseHistory-derived features: {_history_importance:.4f} "
      f"({100 * _history_importance / importances.sum():.1f}% of total)")

_task_prefixes = ('num_tasks', 'num_high_priority_tasks', 'mean_task_call_duration_sec',
                   'max_task_call_duration_sec', 'sum_task_call_duration_sec', 'num_tasks_type_')
_task_importance = importances[importances.index.str.startswith(_task_prefixes)].sum()
print(f"Total importance from Task-derived features: {_task_importance:.4f} "
      f"({100 * _task_importance / importances.sum():.1f}% of total)")

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

result_row = {
    'Model': 'XGBoost',
    'KG': 'N/A (tabular baseline)',
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
    'F1': f1_score(y_test, y_pred),
    'PR-AUC': average_precision_score(y_test, y_pred_proba),
    'Class 0 F1': f1_score(y_test, y_pred, pos_label=0),
    'Class 1 F1': f1_score(y_test, y_pred, pos_label=1),
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'Num Parameters': int(sum(t.count('\n') for t in model.get_booster().get_dump())),
    'Best Epoch': int(model.best_iteration),
    'Runtime (s)': runtime_sec,
    'Classifier': 'XGBoost (tabular baseline, direct fit)',
    'Status': 'ok',
}

results_path = f"{AUTOML_DIR}tabular_baseline_results.csv"
pd.DataFrame([result_row]).to_csv(results_path, index=False)
print(f"Saved: {results_path}")
print(pd.DataFrame([result_row]).to_string(index=False))